# Phase 2: Clean Data & Reconstruct Conversation Threads (AmazonHelp)

### Objective
Transform disconnected tweet records into clean, structured, multi-turn conversation objects:
```json
{
  "thread_id": "12345",
  "brand": "AmazonHelp",
  "turns_count": 2,
  "messages": [
    {"speaker": "customer", "tweet_id": 12345, "text": "Where is my order #..."},
    {"speaker": "brand", "tweet_id": 12346, "text": "Hi there! You can track your shipment here..."}
  ],
  "resolution": "...",
  "resolved": true / false
}
```

### Why This Is Essential
- TWCS tweets arrive out of order, interleaving thousands of different conversations.
- Customer intent is initiated at the root tweet, but the **resolution** and **grounding evidence** only exist at later turns.
- Critical Rule: **Do NOT assume every thread is resolved!** Most Twitter support interactions end in DM redirection, customer silence, or unresolved frustration. Treating every interaction as a 'successful reference resolution' will poison your retrieval database.

In [ ]:
import os
import sys
# Add src to pythonpath
sys.path.append(os.path.abspath("../src"))

from conversations import extract_brand_tweets, reconstruct_threads, save_threads
import pandas as pd
import json

raw_csv = "../data/raw/twcs.csv"
brand_csv = "../data/processed/amazon_raw.csv"
threads_json = "../data/processed/amazon_threads.json"

# Step 1: Extract AmazonHelp tweets (capped at 80,000 for sub-15 minute execution)
extract_brand_tweets(raw_csv, brand_csv, brand_handle="AmazonHelp", max_tweets=80000)

In [ ]:
# Step 2: Load and reconstruct threads
df_brand = pd.read_csv(brand_csv, low_memory=False)
threads = reconstruct_threads(df_brand, brand_handle="AmazonHelp")
save_threads(threads, threads_json)

print(f"Reconstructed {len(threads):,} valid customer-brand conversation threads.")
resolved_count = sum(1 for t in threads if t["resolved"])
print(f"Resolved threads: {resolved_count:,} ({resolved_count/len(threads)*100:.1f}%)")
print(f"Unresolved / DM deflected threads: {len(threads) - resolved_count:,} ({(len(threads)-resolved_count)/len(threads)*100:.1f}%)")

In [ ]:
# Step 3: Inspect 15-20 Sample Reconstructed Threads
print("=== 15 SAMPLE RECONSTRUCTED THREADS ===\n")
for i, thread in enumerate(threads[:18]):
    status_tag = "✅ RESOLVED" if thread["resolved"] else "❌ UNRESOLVED / DM DEFLECTED"
    print(f"--- Thread #{i+1} [ID: {thread['thread_id']}] | {status_tag} | Turns: {thread['turns_count']} ---")
    for msg in thread["messages"]:
        print(f"  [{msg['speaker'].upper()}]: {msg['text']}")
    print(f"  -> Outcome Note: {thread['resolution']}\n")